# LEAF on an A100: latency sessions for the KBS revision

Guide: `scripts/colab/README.md`. Runtime: **A100 GPU** (Colab Pro/Pro+ or any single-A100 machine), nothing else running on the card.

Phases (set `PHASES` below):
- `transfer` (~1 h): the sealed latency-pricing arms (SALT-planned on the home GPU, GMAC-planned, DiffRate, dense) timed at batch 4 and 32 -> `second_device_transfer.tar.gz` (home: `bash scripts/run_second_device.sh home-transfer <archive>`).
- `main` (~1.5-2 h): every arm of the main comparison for DeiT-S/B and ViT-B (MAE) timed in one session (batch 32 for all, 4/128 for LEAF, DiffRate, dense) -> `a100_timing_<device>.tar.gz` (home: `bash scripts/a100_home.sh <archive>`).
- `diffrate384` (optional, ~8-11 h per budget): DiffRate's own rate search on ViT-B/16-384; needs the ImageNet-1k train shards on local disk (about 150 GB) and a Hugging Face token.

Inputs: the kit `a100_kit.tar.gz` (made at home by `python leaf/reproduction/experiments/a100_timing.py export`), uploaded here or placed on Drive. Outputs are copied to `OUT_DIR`.

In [ ]:
PHASES = ['transfer', 'main']          # add 'diffrate384' only with the train shards available (see the last section)
BRANCH = 'dev'
REPO_URL = 'https://github.com/huynhcanhvien/LEAF.git'
KIT = '/content/a100_kit.tar.gz'       # upload below, or point to a Drive path such as /content/drive/MyDrive/leaf/a100_kit.tar.gz
OUT_DIR = '/content/drive/MyDrive/leaf_a100'   # where the result archives are copied (Drive is mounted below); falls back to /content/out
DIFFRATE_BUDGETS = ['27.1', '35.2']    # GMAC of the LEAF 384 schedules at 0.56x and 0.72x of the dense latency
DIFFRATE_MICRO = 64                    # micro-batch at 384 px: 64 fits an A100-40GB, 128 an A100-80GB

In [ ]:
!nvidia-smi
!nvidia-smi --query-compute-apps=pid,gpu_bus_id --format=csv
import torch; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0), 'bf16', torch.cuda.is_bf16_supported())
assert 'A100' in torch.cuda.get_device_name(0) or 'H100' in torch.cuda.get_device_name(0), 'choose an A100 (or H100) runtime'
assert torch.cuda.device_count() == 1, 'timing needs one card with nothing else on it'

In [ ]:
import os, subprocess, shutil
try:
    from google.colab import drive; drive.mount('/content/drive'); os.makedirs(OUT_DIR, exist_ok=True)
except Exception as e:
    print('no Drive:', e); OUT_DIR = '/content/out'; os.makedirs(OUT_DIR, exist_ok=True)
if not os.path.exists(KIT):
    from google.colab import files
    up = files.upload(); KIT = '/content/' + next(iter(up))
print('kit', KIT, os.path.getsize(KIT) // 1000, 'kB; outputs ->', OUT_DIR)

In [ ]:
%cd /content
!rm -rf /content/LEAF && git clone -q -b $BRANCH $REPO_URL /content/LEAF
%cd /content/LEAF
!git log --oneline -1
!pip install -q 'timm==1.0.30' scipy python-dotenv
!pip install -q -e . --no-deps
!tar -xzf $KIT -C /content/LEAF && ls -la results/a100_timing leaf/reproduction/experiments/a100_timing.py
import triton, timm; print('triton', triton.__version__, 'timm', timm.__version__)

## Phase `transfer`: the sealed pricing arms on this GPU (batch 4 and 32)
Same script as the T4 study (`scripts/run_second_device.sh device-transfer`), in BF16 since the A100 supports it. The dense model is timed first and last within every round, so session drift is visible in the archive.

In [ ]:
import os, subprocess, glob, shutil
if 'transfer' in PHASES:
    env = dict(os.environ, LEAF_DTYPE='bfloat16', CUDA_VISIBLE_DEVICES='0')
    r = subprocess.run(['bash', 'scripts/run_second_device.sh', 'device-transfer', 'scripts/kaggle/second_device_kit.tar.gz'], env=env)
    print('exit', r.returncode)
    for f in glob.glob('/content/LEAF/second_device_transfer*.tar.gz'): shutil.copy(f, OUT_DIR); print('saved', f, '->', OUT_DIR)

## Phase `main`: every arm of the main comparison in one session
A three-arm smoke test first (builds a LEAF, a DiffRate and a dense model), then the full session: about 85 arms per backbone, three interleaved rounds.

In [ ]:
import os, subprocess, glob, shutil
if 'main' in PHASES:
    env = dict(os.environ, LEAF_DTYPE='bfloat16', CUDA_VISIBLE_DEVICES='0')
    A = 'leaf/reproduction/experiments/a100_timing.py'
    assert subprocess.run(['python', A, 'smoke'], env=env).returncode == 0, 'smoke test failed; see the output above'
    r = subprocess.run(['python', A, 'time'], env=env); print('exit', r.returncode)
    subprocess.run(['python', A, 'pack'], env=env)
    for f in glob.glob('/content/LEAF/a100_timing_*.tar.gz'): shutil.copy(f, OUT_DIR); print('saved', f, '->', OUT_DIR)

## Optional phase `diffrate384`: DiffRate's own search on ViT-B/16-384
Official DiffRate code (fetched at the pinned commit), 3 epochs over the ImageNet-1k training set per budget, batch 1024 by accumulation. The loader reads WebDataset shards `imagenet1k-train-*.tar` from `data/imagenet_tar`; download them from `timm/imagenet-1k-wds` (about 150 GB, a Hugging Face token with access to the dataset) or mount a Drive copy. One budget takes roughly 8-11 h on an A100; the search checkpoints every few hundred steps and resumes if the session restarts (keep `results/diffrate_vitb384_search` on Drive).

In [ ]:
import os, subprocess, glob, shutil
if 'diffrate384' in PHASES:
    HF_TOKEN = os.environ.get('HF_TOKEN') or input('Hugging Face token (read access to timm/imagenet-1k-wds): ')
    SHARDS = '/content/LEAF/data/imagenet_tar'; os.makedirs(SHARDS, exist_ok=True)
    if len(glob.glob(SHARDS + '/imagenet1k-train-*.tar')) < 1024:
        from huggingface_hub import snapshot_download
        snapshot_download('timm/imagenet-1k-wds', repo_type='dataset', allow_patterns=['imagenet1k-train-*.tar'], local_dir=SHARDS, token=HF_TOKEN, max_workers=8)
    print(len(glob.glob(SHARDS + '/imagenet1k-train-*.tar')), 'train shards')
    res = '/content/LEAF/results/diffrate_vitb384_search'; keep = os.path.join(OUT_DIR, 'diffrate_vitb384_search')
    if os.path.isdir(keep) and not os.path.isdir(res): shutil.copytree(keep, res)      # resume from Drive
    env = dict(os.environ, CUDA_VISIBLE_DEVICES='0', HF_TOKEN=HF_TOKEN)
    subprocess.run(['python', 'leaf/reproduction/experiments/diffrate_clip_search.py', 'fetch'], env=env, check=True)
    D = 'leaf/reproduction/experiments/diffrate_vitb384_search.py'
    assert subprocess.run(['python', D, 'search', '--budgets', DIFFRATE_BUDGETS[0], '--max-steps', '2', '--micro', str(DIFFRATE_MICRO)], env=env).returncode == 0, 'smoke test failed'
    for b in DIFFRATE_BUDGETS:
        subprocess.run(['python', D, 'search', '--budgets', b, '--micro', str(DIFFRATE_MICRO)], env=env)
        shutil.copytree(res, keep, dirs_exist_ok=True); print('checkpointed to', keep)
    subprocess.run(['python', D, 'rates'], env=env); shutil.copytree(res, keep, dirs_exist_ok=True)